# A Lean-Verified Adjustment Theorem

This notebook states a theorem for the “adjustment method” suggested by the tradeoff experiments. The intended use is:

1. Start from an existing training curve or baseline schedule.
2. Propose an adjusted upstream/downstream tradeoff.
3. Measure the immediate or prefix loss penalty of the adjustment.
4. Estimate the future contraction gain from the Jacobian.
5. Certify improvement when the future gain is larger than the prefix penalty.

## Log-Domain Form

The exponential predictor compares an adjusted candidate with a baseline by the ratio

$$
\frac{\widehat L_{\mathrm{adjusted}}}
     {\widehat L_{\mathrm{baseline}}}
=
\text{prefixRatio}\cdot \exp(-\text{totalGain}).
$$

Taking logs gives

$$
\log
\frac{\widehat L_{\mathrm{adjusted}}}
     {\widehat L_{\mathrm{baseline}}}
=
\text{prefixPenalty}-\text{totalGain}.
$$

Therefore, a sufficient and necessary log-domain certificate for predicted improvement is

$$
\text{prefixPenalty}<\text{totalGain}.
$$

## Adjustment Theorem

Consider a set of checkpoints along a training curve. At each checkpoint $t$, compare a proposed adjusted tradeoff against the current baseline. Let

$$
\operatorname{penalty}(t)
$$

be the log prefix-loss penalty, and let

$$
\operatorname{gain}(t)
$$

be the predicted future contraction gain. If

$$
\operatorname{penalty}(t)<\operatorname{gain}(t)
\qquad
\text{for every checkpoint }t,
$$

then the adjusted curve has smaller predicted loss than the baseline at every checkpoint:

$$
\log
\frac{\widehat L_{\mathrm{adjusted}}(t)}
     {\widehat L_{\mathrm{baseline}}(t)}
<0.
$$

This is the formal version of the adjustment principle: a tradeoff adjustment is guaranteed to improve the predicted curve whenever its future geometric gain dominates its current loss cost at every point where we apply it.

## Lean4 Verification

The following proof is checked by Lean 4.32 without Mathlib. To avoid depending on Mathlib’s real-analysis cache, it proves the log-domain algebra using integer-valued log units. This is the exact algebraic core of the real-valued theorem; the real version replaces `Int` by real numbers and uses the standard order-preserving properties of `log` and `exp`.

**Lean status.** The displayed Lean code is written in Lean 4 core style and was checked on the remote server with Lean 4.32.0 using `lean <file>.lean`.


```lean
/-
Lean 4.32 core-verified log-domain tradeoff certificates.

The real-valued exponential predictor compares candidate b with baseline a:

  predicted_ratio = prefix_ratio * exp (- total_gain).

Taking logarithms gives the equivalent log-domain condition:

  log(predicted_ratio) = prefix_penalty - total_gain.

Thus predicted_ratio < 1 is certified by

  prefix_penalty < total_gain.

This file formalizes the log-domain algebra.  The real-analysis facts about
log and exp are standard; using this log form avoids a heavy Mathlib cache
dependency while still machine-checking the decision rule used by the notebooks.
-/

def logPredictedRatio (prefixPenalty totalGain : Int) : Int :=
  prefixPenalty - totalGain

theorem log_tradeoff_certificate
    {prefixPenalty totalGain : Int}
    (h : prefixPenalty < totalGain) :
    logPredictedRatio prefixPenalty totalGain < 0 := by
  unfold logPredictedRatio
  exact Int.sub_neg_of_lt h

def accumulatedGain3 (g1 g2 g3 : Int) : Int :=
  g1 + g2 + g3

theorem accumulated_log_tradeoff_certificate
    {prefixPenalty g1 g2 g3 : Int}
    (h : prefixPenalty < accumulatedGain3 g1 g2 g3) :
    logPredictedRatio prefixPenalty (accumulatedGain3 g1 g2 g3) < 0 := by
  exact log_tradeoff_certificate h

def firstOrderLossRatio (rho : Int) : Int :=
  1 - 2 * rho

theorem positive_rate_improves_first_order_loss
    {rho : Int}
    (hrho : 0 < rho) :
    firstOrderLossRatio rho < 1 := by
  unfold firstOrderLossRatio
  omega

def pointwiseImproves {n : Nat} (penalty gain : Fin n -> Int) : Prop :=
  forall t : Fin n, logPredictedRatio (penalty t) (gain t) < 0

theorem adjustment_method_pointwise_improves
    {n : Nat}
    {penalty gain : Fin n -> Int}
    (hcert : forall t : Fin n, penalty t < gain t) :
    pointwiseImproves penalty gain := by
  intro t
  exact log_tradeoff_certificate (hcert t)

```

## What This Does and Does Not Prove

This theorem does not say that every proposed adjustment will improve real training. It says something sharper and safer:

If the measured/predicted quantities satisfy the certificate at the checkpoints, then the local exponential model forces the adjusted predicted curve to be better.

The experimental notebooks are still responsible for testing whether the local model is accurate for a given architecture, step size, and horizon.